In [1]:
# Imports
import importlib
import random
from collections import Counter
import lm.preprocessing as preprocessing
import json
from package_submission import write_manifest
from config import (
    TRAIN_DIR,
    VALID_DIR,
    TEST_DIR,
    PROCESSED_DIR,
    RESULT_DIR,
    SEED,
    TRAIN_TOKEN_LIMIT,
    VALID_TOKEN_LIMIT,
    TEST_TOKEN_LIMIT,
)
importlib.reload(preprocessing)

<module 'lm.preprocessing' from '/home/gokay/PycharmProjects/nlp_homework/lm/preprocessing.py'>

In [2]:
# --- Train Data ---
train_files = sorted(TRAIN_DIR.glob("*.txt"))
print("Train data size:", len(train_files))
with train_files[0].open(encoding="utf-8") as file:
    print(file.read(1000))

Train data size: 28602














The Old Testament of the King James Version of the Bible




The First Book of Moses:  Called Genesis


1:1 In the beginning God created the heaven and the earth.

1:2 And the earth was without form, and void; and darkness was upon
the face of the deep. And the Spirit of God moved upon the face of the
waters.

1:3 And God said, Let there be light: and there was light.

1:4 And God saw the light, that it was good: and God divided the light
from the darkness.

1:5 And God called the light Day, and the darkness he called Night.
And the evening and the morning were the first day.

1:6 And God said, Let there be a firmament in the midst of the waters,
and let it divide the waters from the waters.

1:7 And God made the firmament, and divided the waters which were
under the firmament from the waters which were above the firmament:
and it was so.

1:8 And God called the firmament Heaven. And the evening and the
morning were the second day.

1:9 And God said

In [3]:
# --- Test Data ---
test_files = sorted(TEST_DIR.glob("*.txt"))
print("Test data size:", len(test_files))
with test_files[0].open(encoding="utf-8") as file:
    print(file.read(1000))

Test data size: 100
ST. PAUL***


E-text prepared by Josephine Paolucci and the Project Gutenberg Online
Distributed Proofreading Team



REMINISCENCES OF PIONEER DAYS IN ST. PAUL

A Collection of Articles Written for and Published in the Daily
Pioneer Press.

By FRANK MOORE






NEWSPAPER STRUGGLES OF PIONEER DAYS.

A BRIEF NARRATION OF INCIDENTS AND EVENTS CONNECTED WITH THE EARLY
DAYS OF ST. PAUL, DAILY NEWSPAPERS.


If James M. Goodhue could revisit the earth and make a tour among the
daily newspaper offices of St. Paul he would discover that wonderful
strides had been made in the method of producing a newspaper during
the latter half of the past century. Among the first things to attract
the attention of this old-timer would be the web-perfecting press,
capable of producing 25,000 impressions an hour, instead of the old
hand press of 240 impressions an hour; the linotype machine, capable
of setting 6,000 to 10,000 ems per hour, instead of the old hand
compositor producing only 80

In [4]:
# --- Validation Data ---
valid_files = sorted(VALID_DIR.glob("*.txt"))
print("Validation data size:", len(valid_files))
with valid_files[0].open(encoding="utf-8") as file:
    print(file.read(1000))

Validation data size: 50




Produced by Q Myers





WALKING

by Henry David Thoreau


I wish to speak a word for Nature, for absolute freedom and wildness, as
contrasted with a freedom and culture merely civil--to regard man as
an inhabitant, or a part and parcel of Nature, rather than a member
of society. I wish to make an extreme statement, if so I may make
an emphatic one, for there are enough champions of civilization: the
minister and the school committee and every one of you will take care of
that.



I have met with but one or two persons in the course of my life who
understood the art of Walking, that is, of taking walks--who had a
genius, so to speak, for SAUNTERING, which word is beautifully derived
"from idle people who roved about the country, in the Middle Ages, and
asked charity, under pretense of going a la Sainte Terre," to the Holy
Land, till the children exclaimed, "There goes a Sainte-Terrer," a
Saunterer, a Holy-Lander. They who never go to the Holy Land in their


In [5]:
def select_subset(file_paths, token_limit, seed):
    # Sorting + constant seed, allows to pick same books.
    shuffled_files = sorted(file_paths)
    random.Random(seed).shuffle(shuffled_files)
    documents = []
    total_tokens = 0
    for file_path in shuffled_files:
        if total_tokens >= token_limit:
            break
        with file_path.open(encoding="utf-8") as file:
            text = file.read()
        # Cleaning
        text = preprocessing.clean_text(text)
        # Tokenizing
        tokens = preprocessing.tokenize(text)
        remaining = token_limit - total_tokens
        tokens = tokens[:remaining]
        if not tokens:
            continue
        documents.append({
            "book_id": file_path.stem,
            "tokens": tokens,
        })
        total_tokens += len(tokens)
    return documents

In [6]:
# Train
train_documents = select_subset(
    train_files, TRAIN_TOKEN_LIMIT, SEED
)
# Validation
valid_documents = select_subset(
    valid_files, VALID_TOKEN_LIMIT, SEED
)
# Test
test_documents = select_subset(
    test_files, TEST_TOKEN_LIMIT, SEED
)
# Sizes of each part
for name, documents in [
    ("Train", train_documents),
    ("Validation", valid_documents),
    ("Test", test_documents),
]:
    token_count = sum(len(doc["tokens"]) for doc in documents)
    print(
        f"{name}: {len(documents)} book, "
        f"{token_count:,} tokens"
    )

Train: 20 book, 1,000,000 tokens
Validation: 1 book, 100,000 tokens
Test: 3 book, 100,000 tokens


In [7]:
word_counts = Counter()

for document in train_documents:
    word_counts.update(document["tokens"])

print("Total number of training tokens:", sum(word_counts.values()))
print("Unique token:", len(word_counts))
print("Most common 20 tokens:")
print(word_counts.most_common(20))

Total number of training tokens: 1000000
Unique token: 39625
Most common 20 tokens:
[(',', 66844), ('the', 54193), ('.', 46647), ('of', 32274), ('and', 25916), ('to', 20641), ('a', 17499), ('in', 15482), ('"', 15150), ('-', 13483), ('that', 9097), ('he', 8763), ('it', 8403), ('was', 8287), ('his', 8241), ('is', 6940), ('with', 6203), ('as', 5908), ('for', 5745), (';', 5592)]


In [8]:
MIN_FREQ = 2
MAX_VOCAB_SIZE = 20_000

special_tokens = {"<unk>", "<bos>", "<eos>"}
normal_token_limit = MAX_VOCAB_SIZE - len(special_tokens)
# Get tokens which appear more than 2 times
vocab = {
    token
    for token, count in word_counts.most_common(normal_token_limit)
    if count >= MIN_FREQ
}
vocab.update(special_tokens)
print("Vocabulary size:", len(vocab))

Vocabulary size: 20000


In [9]:
total_tokens = sum(word_counts.values())
unknown_count = sum(
    count
    for token, count in word_counts.items()
    if token not in vocab
)

print("Number of tokens which will be <unk> in Train: ", unknown_count)
print(f"<unk> ratio: %{100 * unknown_count / total_tokens:.2f}")

Number of tokens which will be <unk> in Train:  22760
<unk> ratio: %2.28


In [10]:
importlib.reload(preprocessing)
# Example tokens for replacing unknown tokens
example_vocab = {"the", "cat", "<unk>"}
example_tokens = ["the", "purple", "cat"]

print(preprocessing.replace_unknown(example_tokens, example_vocab))

['the', '<unk>', 'cat']


In [11]:
def prepare_documents(documents, vocab):
    return [
        {
            "book_id": document["book_id"],
            "tokens": preprocessing.replace_unknown(
                document["tokens"], vocab
            ),
        }
        for document in documents
    ]
# Final version of datasets
train_ready = prepare_documents(train_documents, vocab)
valid_ready = prepare_documents(valid_documents, vocab)
test_ready = prepare_documents(test_documents, vocab)

In [12]:
# Output directory configuration
OUTPUT_DIR = PROCESSED_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "train": train_ready,
    "validation": valid_ready,
    "test": test_ready,
}

for name, documents in datasets.items():
    output_path = OUTPUT_DIR / f"{name}.json"
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(documents, file, ensure_ascii=False)
with (OUTPUT_DIR / "vocab.json").open("w", encoding="utf-8") as file:
    json.dump(sorted(vocab), file, ensure_ascii=False)
print("Data and vocabulary are saved to:", OUTPUT_DIR.resolve())

manifest = write_manifest(
    PROCESSED_DIR,
    RESULT_DIR / "data_selection.json",
    seed=SEED,
    token_limits={
        "train": TRAIN_TOKEN_LIMIT,
        "validation": VALID_TOKEN_LIMIT,
        "test": TEST_TOKEN_LIMIT,
    },
    min_freq=MIN_FREQ,
    max_vocab_size=MAX_VOCAB_SIZE,
)

print("Data selection log:", RESULT_DIR / "data_selection.json")

Data and vocabulary are saved to: /home/gokay/PycharmProjects/nlp_homework/data/processed
Data selection log: /home/gokay/PycharmProjects/nlp_homework/results/data_selection.json
